# AgroSense - Phase 1: data pipeline + image-only baseline
Thin driver: all logic lives in `src/`. Every number printed here is read from files written by real runs.

**Before running on Kaggle:** Settings -> Accelerator = GPU (T4/P100), Internet = ON (needed once to download ImageNet weights),
and add both datasets (PlantVillage, PlantDoc) via *Add Input*.

In [ ]:
import os, sys, json
from pathlib import Path
# Make `src` importable whether this runs from notebooks/ (local) or from the repo root (Kaggle working dir).
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)

import pandas as pd
from IPython.display import Image as IPyImage, display
from src.utils import load_config, apply_smoke_overrides, results_dir
from src.data import (resolve_dataset_roots, print_dataset_structure, prepare_data, build_shifted_test,
                      analyse_data, plot_shift_grid, plantdoc_class_folders)
from src.train import train_one_seed
from src.evaluate import evaluate_run, aggregate_runs, format_table

CONFIG = os.environ.get("AGROSENSE_CONFIG", "configs/baseline.yaml")
cfg = load_config(CONFIG)
print("config:", CONFIG, "| experiment:", cfg["experiment_name"], "| seeds:", cfg["seeds"], "| image size:", cfg["data"]["image_size"])

## 1. Inspect the real dataset structure (no guessing)

In [ ]:
pv_root, pd_root = resolve_dataset_roots(cfg)
print_dataset_structure(pv_root, "PlantVillage")
print()
for sd in cfg["data"]["plantdoc_dirs"]:
    print_dataset_structure(pd_root / sd, f"PlantDoc/{sd}")

## 2. Class intersection, near-duplicate groups and leakage-safe split
The mapping table below is computed from the folder names above. **Verify it by eye** - especially the alias lines in the config.

In [ ]:
bundle = prepare_data(cfg)
print(); print(json.dumps(bundle.dup_report, indent=2))

## 3. Data analysis: class balance and the PlantVillage -> PlantDoc shift

In [ ]:
shifted_df, _ = build_shifted_test(bundle, cfg)     # builds the file list only (no training use)
tab = analyse_data(bundle, shifted_df, cfg)
plot_shift_grid(bundle, shifted_df, cfg)
display(tab)
print(open(results_dir(cfg) / "imbalance.json").read())
for f in ("class_distribution.png", "shift_grid.png"):
    p = results_dir(cfg) / f
    if p.exists(): display(IPyImage(str(p)))

## 4. Smoke test (2 epochs, ~200 images, small images; results go to `results/smoke/`)
Proves the pipeline runs end to end. Its metrics are meaningless.

In [ ]:
smoke_cfg = apply_smoke_overrides(cfg)
# If you are offline, set smoke_cfg["model"]["pretrained"] = False for this cell only.
smoke_bundle = prepare_data(smoke_cfg, verbose=False)
train_one_seed(smoke_cfg, cfg["seeds"][0], smoke_bundle)
evaluate_run(smoke_cfg, cfg["seeds"][0], smoke_bundle)

## 5. Full training: seeds 42, 123, 456 (ce_weighted loss)
Checkpoint + early stopping use **ID validation macro F1 only**.

In [ ]:
for seed in cfg["seeds"]:
    train_one_seed(cfg, seed, bundle)

## 6. Evaluate each seed on the ID test set and the shifted (PlantDoc) set
Every shifted-set evaluation is appended to `results/shifted_access_log.csv`.

In [ ]:
for seed in cfg["seeds"]:
    evaluate_run(cfg, seed, bundle)

## 7. Results table (mean +/- std over seeds; sample std, ddof=1)
`target`/`met` compare the **mean** with the project targets; missing a target is a finding to explain, not a bug.

In [ ]:
table = aggregate_runs(cfg)
print(format_table(table))
print("\nSaved:", results_dir(cfg) / f"summary_{cfg['experiment_name']}.csv")
print("Per-seed JSON:", sorted(str(p) for p in (results_dir(cfg) / "runs" / cfg["experiment_name"]).glob("seed_*/metrics.json")))

## Notes for the report
* High-cost class list is a **proposal** (see config) - confirm it with a written justification.
* ECE here is *uncalibrated* (no temperature scaling yet; Phase 4).
* Class weights are inverse-frequency from the training split.